# Day 4, Task 1: fine-tune clause classifiers on CUAD (Colab GPU)

Trains three small models on the CUAD clause dataset built by `python -m src.contracts.cuad_data`
(28 clause types, multi-label, split by contract) and predicts the test set.

1. **Runtime > Change runtime type > T4 GPU**, then run the cells in order.
2. When asked, upload `cuad_clauses_for_colab.zip` (in the project's `data/cuad_labels/`).
3. Download `colab_clause_results.zip` at the end and unzip it into the project's `data/cuad_labels/`.
4. Locally: `python -m evaluations.clause_classification --no-llm` (or without `--no-llm` once the LLM has run)
   compares the LLM with every model.

Each model keeps the epoch with the best validation micro F1, and one decision threshold tuned on validation.
The test contracts are never used for any choice.

In [2]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
print("GPU available:", torch.cuda.is_available())

name, memory.total [MiB]
Tesla T4, 15360 MiB
GPU available: True


In [3]:
from google.colab import files
import os, zipfile
uploaded = files.upload()          # choose cuad_clauses_for_colab.zip
zipfile.ZipFile(next(iter(uploaded))).extractall("data")
print(sorted(os.listdir("data")))

Saving cuad_clauses_for_colab.zip to cuad_clauses_for_colab.zip
['clause_types.json', 'clauses_test.jsonl', 'clauses_test_sample.jsonl', 'clauses_train.jsonl', 'clauses_val.jsonl']


In [4]:
import copy, json, time
import numpy as np, torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup

def load(split):
    return [json.loads(line) for line in open(f"data/clauses_{split}.jsonl", encoding="utf-8")]

TYPES = list(json.load(open("data/clause_types.json", encoding="utf-8")))
train_rows, val_rows, test_rows, sample_rows = load("train"), load("val"), load("test"), load("test_sample")
print(len(train_rows), len(val_rows), len(test_rows), len(sample_rows), "examples;", len(TYPES), "clause types")
device = "cuda" if torch.cuda.is_available() else "cpu"
os.makedirs("out", exist_ok=True)


def micro_f1(probs, rows, threshold):
    gold = np.array([[t in r["labels"] for t in TYPES] for r in rows])
    pred = probs >= threshold
    tp, fp, fn = (pred & gold).sum(), (pred & ~gold).sum(), (~pred & gold).sum()
    return 2 * tp / (2 * tp + fp + fn) if tp else 0.0


@torch.no_grad()
def probabilities(model, tok, rows, max_len, batch=64):
    model.eval()
    out = []
    for i in range(0, len(rows), batch):
        enc = tok([r["text"] for r in rows[i:i + batch]], truncation=True, max_length=max_len,
                  padding=True, return_tensors="pt").to(device)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=device == "cuda"):
            out.append(torch.sigmoid(model(**enc).logits.float()).cpu().numpy())
    return np.concatenate(out)


def train(name, model_name, epochs, lr, batch=16, max_len=256, seed=42):
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)
    tok = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=len(TYPES), problem_type="multi_label_classification",
        id2label=dict(enumerate(TYPES)), label2id={t: i for i, t in enumerate(TYPES)},
        ignore_mismatched_sizes=True, dtype=torch.float32).to(device)   # DeBERTa-v3 is stored in FP16
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    steps = epochs * ((len(train_rows) + batch - 1) // batch)
    sched = get_linear_schedule_with_warmup(opt, int(0.06 * steps), steps)
    scaler = torch.amp.GradScaler(enabled=device == "cuda")
    best_f1, best_state, best_epoch, history, start = -1.0, None, 0, [], time.time()
    for epoch in range(epochs):
        model.train()
        order = rng.permutation(len(train_rows))
        for i in range(0, len(order), batch):
            rows = [train_rows[j] for j in order[i:i + batch]]
            enc = tok([r["text"] for r in rows], truncation=True, max_length=max_len,
                      padding=True, return_tensors="pt").to(device)
            y = torch.tensor([[1.0 if t in r["labels"] else 0.0 for t in TYPES] for r in rows], device=device)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=device == "cuda"):
                logits = model(**enc).logits
            loss = torch.nn.functional.binary_cross_entropy_with_logits(logits.float(), y)
            opt.zero_grad()
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()
            sched.step()
        val_probs = probabilities(model, tok, val_rows, max_len)
        f1 = max(micro_f1(val_probs, val_rows, t) for t in np.arange(0.1, 0.91, 0.05))
        history.append(round(float(f1), 4))
        print(f"{name} epoch {epoch + 1}: validation micro F1 {f1:.3f} ({(time.time() - start) / 60:.1f} min)")
        if f1 > best_f1:
            best_f1, best_state, best_epoch = f1, copy.deepcopy(model.state_dict()), epoch + 1
    model.load_state_dict(best_state)
    val_probs = probabilities(model, tok, val_rows, max_len)
    threshold = float(max(np.arange(0.1, 0.91, 0.05), key=lambda t: micro_f1(val_probs, val_rows, t)))
    meta = {"name": name, "model": model_name, "epochs_run": epochs, "best_epoch": best_epoch, "lr": lr,
            "max_len": max_len, "threshold": round(threshold, 2),
            "val_micro_f1": round(float(micro_f1(val_probs, val_rows, threshold)), 4), "val_history": history,
            "train_minutes": round((time.time() - start) / 60, 1),
            "gpu": torch.cuda.get_device_name(0) if device == "cuda" else "cpu"}
    for split, rows in (("test", test_rows), ("test_sample", sample_rows)):
        t0 = time.time()
        probs = probabilities(model, tok, rows, max_len)
        meta[f"ms_per_passage_{split}"] = round(1000 * (time.time() - t0) / len(rows), 1)
        with open(f"out/pred_ft_{name}_{split}.jsonl", "w", encoding="utf-8") as f:
            for r, p in zip(rows, probs):
                f.write(json.dumps({"id": r["id"], "labels": [t for t, v in zip(TYPES, p) if v >= threshold]}) + "\n")
    json.dump(meta, open(f"out/meta_ft_{name}.json", "w"), indent=1)
    print(meta)
    del model
    torch.cuda.empty_cache()

10088 1218 1054 482 examples; 28 clause types


Three models, about 10-30 minutes in total on a T4. Each cell can be run on its own.

In [5]:
train("legalbert_small", "nlpaueb/legal-bert-small-uncased", epochs=8, lr=5e-5)

config.json:   0%|          | 0.00/989 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/222k [00:00<?, ?B/s]

[transformers] You passed `num_labels=28` which is incompatible to the `id2label` map of length `2`.


pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  141MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: nlpaueb/legal-bert-small-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those

model.safetensors: reconstructing file:   0%|          |  0.00B /  141MB            

model.safetensors: downloading bytes:           |  0.00B            

legalbert_small epoch 1: validation micro F1 0.548 (0.6 min)
legalbert_small epoch 2: validation micro F1 0.710 (1.1 min)
legalbert_small epoch 3: validation micro F1 0.770 (1.6 min)
legalbert_small epoch 4: validation micro F1 0.774 (2.2 min)
legalbert_small epoch 5: validation micro F1 0.790 (2.7 min)
legalbert_small epoch 6: validation micro F1 0.794 (3.2 min)
legalbert_small epoch 7: validation micro F1 0.806 (3.7 min)
legalbert_small epoch 8: validation micro F1 0.813 (4.3 min)
{'name': 'legalbert_small', 'model': 'nlpaueb/legal-bert-small-uncased', 'epochs_run': 8, 'best_epoch': 8, 'lr': 5e-05, 'max_len': 256, 'threshold': 0.3, 'val_micro_f1': 0.8134, 'val_history': [0.5483, 0.7098, 0.7705, 0.7743, 0.7899, 0.7939, 0.8063, 0.8134], 'train_minutes': 4.3, 'gpu': 'Tesla T4', 'ms_per_passage_test': 1.0, 'ms_per_passage_test_sample': 1.0}


In [6]:
train("legalbert_base", "nlpaueb/legal-bert-base-uncased", epochs=5, lr=3e-5)

config.json:   0%|          | 0.00/1.02k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/222k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

[transformers] You passed `num_labels=28` which is incompatible to the `id2label` map of length `2`.


pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  440MB            

... (2 outputs left out)

[transformers] BertForSequenceClassification LOAD REPORT from: nlpaueb/legal-bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those 

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

legalbert_base epoch 1: validation micro F1 0.633 (1.7 min)
legalbert_base epoch 2: validation micro F1 0.752 (3.5 min)
legalbert_base epoch 3: validation micro F1 0.806 (5.2 min)
legalbert_base epoch 4: validation micro F1 0.818 (7.0 min)
legalbert_base epoch 5: validation micro F1 0.827 (8.7 min)
{'name': 'legalbert_base', 'model': 'nlpaueb/legal-bert-base-uncased', 'epochs_run': 5, 'best_epoch': 5, 'lr': 3e-05, 'max_len': 256, 'threshold': 0.3, 'val_micro_f1': 0.8274, 'val_history': [0.6331, 0.7521, 0.8057, 0.8184, 0.8274], 'train_minutes': 8.8, 'gpu': 'Tesla T4', 'ms_per_passage_test': 3.1, 'ms_per_passage_test_sample': 3.0}


In [7]:
train("deberta_v3_base", "microsoft/deberta-v3-base", epochs=5, lr=2e-5)

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model: reconstructing file:   0%|          |  0.00B / 2.46MB            

spm.model: downloading bytes:           |  0.00B            

/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  371MB            

... (2 outputs left out)

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
classifier.weight                       | MISSING    | 
classifier.bias                         | MISSING    | 
pooler.dense.weight                     | MISSING    | 
pooler.den

model.safetensors: reconstructing file:   0%|          |  0.00B /  371MB            

model.safetensors: downloading bytes:           |  0.00B            

deberta_v3_base epoch 1: validation micro F1 0.529 (3.1 min)
deberta_v3_base epoch 2: validation micro F1 0.695 (6.2 min)
deberta_v3_base epoch 3: validation micro F1 0.745 (9.3 min)
deberta_v3_base epoch 4: validation micro F1 0.776 (12.4 min)
deberta_v3_base epoch 5: validation micro F1 0.778 (15.5 min)
{'name': 'deberta_v3_base', 'model': 'microsoft/deberta-v3-base', 'epochs_run': 5, 'best_epoch': 5, 'lr': 2e-05, 'max_len': 256, 'threshold': 0.3, 'val_micro_f1': 0.7783, 'val_history': [0.5291, 0.6955, 0.7451, 0.7759, 0.7783], 'train_minutes': 15.7, 'gpu': 'Tesla T4', 'ms_per_passage_test': 5.7, 'ms_per_passage_test_sample': 5.7}


In [8]:
import shutil
shutil.make_archive("colab_clause_results", "zip", "out")
files.download("colab_clause_results.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>